# 🧠 EXACT 2026 - Track 1: Logic-Based Educational QA
### Hướng dẫn chạy thử nghiệm Pipeline trên Google Colab với Google Drive Cache

Notebook này hướng dẫn bạn thiết lập môi trường và chạy thử nghiệm hệ thống Neuro-Symbolic QA (Track 1) trên Google Colab sử dụng GPU. 

**⚠️ LƯU Ý QUAN TRỌNG:**
Do kiến trúc mới của hệ thống đã chuyển sang mô hình kết nối qua **Inference Server (OpenAI-compatible)** để đáp ứng kiểm tra của Ban tổ chức, bạn **PHẢI khởi chạy server chạy ngầm trước** rồi mới thực hiện chạy pipeline. Script chạy sẽ tự động kết nối qua API ở cổng 8000.


**✨ TÍNH NĂNG MỚI:**
Hệ thống đã hỗ trợ trích xuất đầy đủ và chính xác trường `premises_used` (chiếm 50% điểm số câu hỏi Type 1) từ cả bộ giải ký hiệu (Logic Tree) lẫn bộ giải neural fallback (LLM Chain-of-Thought). Kết quả offline sẽ được tự động chuyển đổi thành chỉ mục 1-based để khớp với nhãn ground truth.

---

## 1. Kết nối Google Drive và kiểm tra GPU

Chạy cell dưới đây để kết nối với Google Drive của bạn (nhằm truy cập thư mục chứa file precompiled wheel và model `Colab_Cache`). Đồng thời kiểm tra thông tin GPU.

*(Lưu ý: Đi tới **Runtime** -> **Change runtime type** -> chọn **T4 GPU** trước khi chạy)*

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Kiểm tra GPU
!nvidia-smi

Mounted at /content/drive
Fri Jun 12 05:59:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

## 2. Clone Repository và chuyển sang nhánh `test/track1`

In [2]:
# Clone repo từ Github
!git clone https://github.com/AIVIETNAM-AIO-Triet-Descartes/EXACT2026-NeuroSymbolic-QA.git

# Chuyển con trỏ dòng lệnh vào thư mục dự án
%cd /content/EXACT2026-NeuroSymbolic-QA

# Chuyển sang nhánh test/track1
!git checkout test/track1

Cloning into 'EXACT2026-NeuroSymbolic-QA'...
remote: Enumerating objects: 842, done.
remote: Counting objects: 100% (274/274), done.
remote: Compressing objects: 100% (187/187), done.
remote: Total 842 (delta 134), reused 194 (delta 82), pack-reused 568 (from 1)
Receiving objects: 100% (842/842), 3.18 MiB | 23.42 MiB/s, done.
Resolving deltas: 100% (473/473), done.
/content/EXACT2026-NeuroSymbolic-QA
Branch 'test/track1' set up to track remote branch 'test/track1' from 'origin'.
Switched to a new branch 'test/track1'


## 3. Cài đặt các thư viện phụ thuộc (Dependencies)

Chúng ta sẽ cài đặt các thư viện trong `requirements.txt`. Riêng đối với `llama-cpp-python`, ta sẽ cài đặt trực tiếp từ file `.whl` đã được biên dịch sẵn trong thư mục `Colab_Cache` trên Google Drive.

In [3]:
# Đảm bảo đứng đúng thư mục dự án và cài đặt dependencies
%cd /content/EXACT2026-NeuroSymbolic-QA

!pip install -r requirements.txt

# Cài đặt llama-cpp-python từ file wheel (.whl) lưu trên Google Drive để tiết kiệm thời gian
!pip install /content/drive/MyDrive/Colab_Cache/llama_cpp_python-0.3.23-py3-none-linux_x86_64.whl

/content/EXACT2026-NeuroSymbolic-QA
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.7/31.7 MB 69.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.9 

Processing /content/drive/MyDrive/Colab_Cache/llama_cpp_python-0.3.23-py3-none-linux_x86_64.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00


## 3.5 Tải Mô hình DeepSeek-R1-0528-Qwen3-8B GGUF vào Google Drive

Nếu trong thư mục `Colab_Cache` trên Drive của bạn chưa có sẵn tệp GGUF của DeepSeek, chạy cell này để tải bản `Q4_K_M` (~4.8GB) trực tiếp từ repo Unsloth trên HuggingFace về thư mục cache của Drive. Quá trình này chỉ cần chạy **một lần duy nhất**.

In [4]:
# Đảm bảo thư mục lưu trữ cache trên Google Drive tồn tại
!mkdir -p /content/drive/MyDrive/Colab_Cache/

# Tải tệp GGUF trực tiếp từ HuggingFace về Drive (hỗ trợ resume tải tiếp nếu bị ngắt quãng)
!wget -c https://huggingface.co/unsloth/DeepSeek-R1-0528-Qwen3-8B-GGUF/resolve/main/DeepSeek-R1-0528-Qwen3-8B-Q4_K_M.gguf -O /content/drive/MyDrive/Colab_Cache/DeepSeek-R1-0528-Qwen3-8B-Q4_K_M.gguf

# Kiểm tra tệp tin
import os
if os.path.exists("/content/drive/MyDrive/Colab_Cache/DeepSeek-R1-0528-Qwen3-8B-Q4_K_M.gguf"):
    print("✅ Mô hình DeepSeek đã được tải xuống và lưu trữ trên Drive!")
else:
    print("❌ Tải mô hình thất bại hoặc chưa hoàn thành.")

--2026-06-11 14:33:05--  https://huggingface.co/unsloth/DeepSeek-R1-0528-Qwen3-8B-GGUF/resolve/main/DeepSeek-R1-0528-Qwen3-8B-Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 13.35.202.40, 13.35.202.97, 13.35.202.121, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.40|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/68386c758c8e3b721707c6ae/d8be09a4e6b490b68df4262691283b3a31e9b25f65203ae7b24862469332df30?Expires=1781191985&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY4Mzg2Yzc1OGM4ZTNiNzIxNzA3YzZhZS9kOGJlMDlhNGU2YjQ5MGI2OGRmNDI2MjY5MTI4M2IzYTMxZTliMjVmNjUyMDNhZTdiMjQ4NjI0NjkzMzJkZjMwKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4MTE5MTk4NX19fV19&Signature=MEUCIC77rnZUnGHHK6H5zLeWG88BLhqs--DdPHEdtKcHfYtBAiEA-A-mT3xB7Xzap18iKwNIpLev%7EnrAjsVB7PpE%7EYBDOcc_&Key-Pair-Id=K1LYXO563TGWFU&X-Xet-Cas-Uid=public&response-c

## 4. Chuẩn bị Mô hình GGUF

Sao chép các file mô hình GGUF (Qwen 2.5 7B hoặc DeepSeek-R1-0528-Qwen3-8B) từ Drive vào ổ đĩa của Colab để tăng tốc độ load và suy luận.

In [4]:
# Di chuyển vào thư mục dự án
%cd /content/EXACT2026-NeuroSymbolic-QA

import shutil
import os

# CHỌN MỘT TRONG HAI MÔ HÌNH DƯỚI ĐÂY (Bỏ comment dòng lệnh tương ứng):

# --- LỰA CHỌN A: Qwen 2.5 7B Instruct GGUF ---
shutil.copy("/content/drive/MyDrive/Colab_Cache/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf", ".")
shutil.copy("/content/drive/MyDrive/Colab_Cache/qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf", ".")

# --- LỰA CHỌN B: DeepSeek-R1-0528-Qwen3-8B GGUF ---
# shutil.copy("/content/drive/MyDrive/Colab_Cache/DeepSeek-R1-0528-Qwen3-8B-Q4_K_M.gguf", ".")

# Kiểm tra file đã tồn tại ở local hay chưa
qwen_exists = os.path.exists("./qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf")
deepseek_exists = os.path.exists("./DeepSeek-R1-0528-Qwen3-8B-Q4_K_M.gguf")
if qwen_exists or deepseek_exists:
    print(f"✅ Copy model thành công (Qwen: {qwen_exists}, DeepSeek: {deepseek_exists})!")
else:
    print("❌ Chưa tìm thấy file model local. Hãy kiểm tra lại dòng lệnh copy.")

/content/EXACT2026-NeuroSymbolic-QA
✅ Copy model thành công (Qwen: True, DeepSeek: False)!


## 5. Khởi động Inference Server chạy ngầm

Chọn một trong hai cách khởi động dưới đây để bật Server chạy ngầm trên cổng `8000` (hỗ trợ cả Qwen 2.5 7B và DeepSeek-R1-0528-Qwen3-8B):

### **Cách 1: Khởi động Server qua `llama-cpp-python` (Khuyên dùng - Sử dụng file GGUF)**

In [6]:
!pip install "llama-cpp-python[server]"
!cat llama_server.log
import os
print(os.listdir(".")) # Xem danh sách file hiện có để đối chiếu tên mô hình


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/llama_cpp/server/__main__.py", line 33, in <module>
    from llama_cpp.server.app import create_app
  File "/usr/local/lib/python3.12/dist-packages/llama_cpp/server/app.py", line 22, in <module>
    from starlette_context.plugins import RequestIdPlugin  # type: ignore
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'starlette_context'
['output', 'evaluation', '.gitignore', 'configs', 'llama_server.log', 'reports', 'docs', '.env.example', 'tests', 'qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf', 'scripts', 'README.md', 'CLAUDE.md', 'logic_dataset_analysis_findings.json', 'logic_dataset_analysis_report.md', 'api', 'requirements.txt', 'pipeline', 'requirements-dev.txt', 'llm', 'LICENSE', 'data', '.git', 'run_track1_colab.ipynb', 'qwen2.5-7b-instruct-q4_k_m-

In [16]:
!cat llama_server.log
!python3 -m llama_cpp.server --model ./DeepSeek-R1-0528-Qwen3-8B-Q4_K_M.gguf --port 8000 --n_gpu_layers -1 --n_ctx 4096 --model_alias DeepSeek-R1-0528-Qwen3-8B


usage: __main__.py [-h] [--model MODEL] [--model_alias MODEL_ALIAS]
                   [--n_gpu_layers N_GPU_LAYERS] [--split_mode SPLIT_MODE]
                   [--main_gpu MAIN_GPU] [--tensor_split [TENSOR_SPLIT ...]]
                   [--vocab_only VOCAB_ONLY] [--use_mmap USE_MMAP]
                   [--use_mlock USE_MLOCK] [--kv_overrides [KV_OVERRIDES ...]]
                   [--rpc_servers RPC_SERVERS] [--seed SEED] [--n_ctx N_CTX]
                   [--n_batch N_BATCH] [--n_ubatch N_UBATCH]
                   [--n_threads N_THREADS] [--n_threads_batch N_THREADS_BATCH]
                   [--rope_scaling_type ROPE_SCALING_TYPE]
                   [--rope_freq_base ROPE_FREQ_BASE]
                   [--rope_freq_scale ROPE_FREQ_SCALE]
                   [--yarn_ext_factor YARN_EXT_FACTOR]
                   [--yarn_attn_factor YARN_ATTN_FACTOR]
                   [--yarn_beta_fast YARN_BETA_FAST]
                   [--yarn_beta_slow YARN_BETA_SLOW]
                   [--yarn_orig_

In [7]:
%cd /content/EXACT2026-NeuroSymbolic-QA

# CHỌN VÀ BỎ COMMENT DÒNG KHỞI CHẠY TƯƠNG ỨNG VỚI MÔ HÌNH ĐÃ DOWNLOAD:

# --- Lựa chọn 1: Chạy Qwen 2.5 7B ---
!nohup python3 -m llama_cpp.server --model ./qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf --port 8000 --n_gpu_layers -1 --n_ctx 4096 --model_alias Qwen/Qwen2.5-7B-Instruct > llama_server.log 2>&1 &
!sed -i 's/model_name: .*/model_name: "Qwen\/Qwen2.5-7B-Instruct"/g' configs/config.yaml

# --- Lựa chọn 2: Chạy DeepSeek-R1-0528-Qwen3-8B ---
# !nohup python3 -m llama_cpp.server --model ./DeepSeek-R1-0528-Qwen3-8B-Q4_K_M.gguf --port 8000 --n_gpu_layers -1 --n_ctx 4096 --model_alias DeepSeek-R1-0528-Qwen3-8B > llama_server.log 2>&1 &
# !sed -i 's/model_name: .*/model_name: "DeepSeek-R1-0528-Qwen3-8B"/g' configs/config.yaml

print("⏳ Đang khởi động Server... Vui lòng đợi khoảng 30 giây.")
import time
time.sleep(30)

# Kiểm tra trạng thái hoạt động của Server
!curl http://localhost:8000/v1/models

/content/EXACT2026-NeuroSymbolic-QA
⏳ Đang khởi động Server... Vui lòng đợi khoảng 30 giây.
{"object":"list","data":[{"id":"Qwen/Qwen2.5-7B-Instruct","object":"model","owned_by":"me","permissions":[]}]}

In [23]:
!fuser -k 8000/tcp

### **Cách 2: Khởi động Server bằng `vLLM` (Tự tải safetensors gốc từ HuggingFace)**
Cách này sẽ tải mô hình chính thức định dạng safetensors trực tiếp từ HuggingFace về bộ nhớ Colab, giúp tăng tốc độ suy luận tối đa.

In [ ]:
# Cài đặt vLLM
!pip install vllm

# CHỌN VÀ BỎ COMMENT MÔ HÌNH CHẠY VỚI VLLM (Tự động tải từ HuggingFace hoặc dùng thư mục local):

# --- Lựa chọn 1: Chạy Qwen 2.5 7B ---
# !nohup vllm serve Qwen/Qwen2.5-7B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.9 --max-model-len 2048 > vllm.log 2>&1 &
# !sed -i 's/model_name: .*/model_name: "Qwen\/Qwen2.5-7B-Instruct"/g' configs/config.yaml

# --- Lựa chọn 2: Chạy DeepSeek-R1-0528-Qwen3-8B ---
# !nohup vllm serve deepseek-ai/DeepSeek-R1-0528-Qwen3-8B --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.9 --max-model-len 2048 > vllm.log 2>&1 &
# !sed -i 's/model_name: .*/model_name: "DeepSeek-R1-0528-Qwen3-8B"/g' configs/config.yaml

print("⏳ Đang khởi động vLLM Server... Quá trình này mất khoảng 2-3 phút.")
import time
time.sleep(120)

# Kiểm tra trạng thái hoạt động của Server
!curl http://localhost:8000/v1/models

## 6. Chạy thử nghiệm Pipeline (Track 1)

Sử dụng script `scripts/run_track1.py` để chạy pipeline. Hệ thống sẽ tự động kết nối qua REST API đang phục vụ ở cổng 8000 để chạy suy luận.

### 6.1 Chạy thử nhanh với 5 mẫu đầu tiên

In [12]:
# Di chuyển vào thư mục dự án trước khi chạy
%cd /content/EXACT2026-NeuroSymbolic-QA

!PYTHONPATH=. python3 scripts/run_track1.py \
    --input Logic_Based_Educational_Queries.json \
    --output output/predictions_test.json \
    --max-samples 50 \
    --evaluate

/content/EXACT2026-NeuroSymbolic-QA
2026-06-12 09:36:49.511 | INFO     | __main__:main:869 - Loading dataset from Logic_Based_Educational_Queries.json...
2026-06-12 09:36:49.522 | INFO     | __main__:main:872 - Loaded 411 samples.
2026-06-12 09:36:49.523 | INFO     | __main__:main:896 - Processing samples from index 0 to 50 (50 samples).
2026-06-12 09:36:49.523 | INFO     | __main__:run:181 - Processing 50 samples...
2026-06-12 09:36:49.531 | INFO     | llm:get_shared_reasoner:52 - [LLM] shared reasoner → http://localhost:8000/v1 (Qwen/Qwen2.5-7B-Instruct)
Processing: 100% 50/50 [26:03<00:00, 31.26s/it]
2026-06-12 10:02:52.648 | INFO     | __main__:_print_stats:683 - ============================================================
2026-06-12 10:02:52.648 | INFO     | __main__:_print_stats:684 - PIPELINE EXECUTION STATISTICS
2026-06-12 10:02:52.648 | INFO     | __main__:_print_stats:685 - ============================================================
2026-06-12 10:02:52.648 | INFO     | __mai

### 6.2 Chạy đánh giá trên dải dữ liệu cụ thể (ví dụ: Mẫu 50 đến 100)

In [ ]:
# Di chuyển vào thư mục dự án trước khi chạy
%cd /content/EXACT2026-NeuroSymbolic-QA

!PYTHONPATH=. python3 scripts/run_track1.py \
    --input Logic_Based_Educational_Queries.json \
    --output output/predictions_50_100.json \
    --start-sample 50 \
    --end-sample 100 \
    --evaluate

### 6.3 Chạy toàn bộ Dataset (411 mẫu / ~808 câu hỏi)

In [ ]:
# Di chuyển vào thư mục dự án trước khi chạy
%cd /content/EXACT2026-NeuroSymbolic-QA

!PYTHONPATH=. python3 scripts/run_track1.py \
    --input Logic_Based_Educational_Queries.json \
    --output output/predictions_full.json \
    --evaluate

## 7. Xem Kết Quả Đầu Ra
Sau khi chạy xong, các kết quả dự đoán và đánh giá chi tiết sẽ được lưu tại thư mục `output/`.

**Lưu ý:** Hãy kiểm tra trường `idx` trong tệp JSON kết quả. Trường này chứa danh sách các tiền đề được sử dụng (`premises_used`) dạng 1-based cho từng câu hỏi con.

In [ ]:
# Di chuyển vào thư mục dự án trước khi chạy
%cd /content/EXACT2026-NeuroSymbolic-QA

# Hiển thị 30 dòng đầu của file dự đoán để kiểm tra cấu trúc đầu ra
!head -n 30 output/predictions_test.json